# 🪸 Coral Bleaching Robustness: 100% Automated GPU Training

**No dataset upload or manual search needed!** This notebook downloads the real NOAA Hawaii Coral Bleaching Dataset (10,419 images) automatically.

### ⚙️ Only 2 settings needed (Right Sidebar):
1. **Accelerator**: Select **GPU T4**
2. **Internet**: Toggle **Internet ON**
3. Click **Run All**!


In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Install required dependencies
!pip install -q timm datasets huggingface_hub roboflow


In [ ]:
%%writefile models.py
"""Models and losses: standard cross-entropy or ordinal (CORAL-style) head."""
import timm
import torch
import torch.nn.functional as F


def build_model(arch, n_classes, loss="ce", pretrained=True):
    n_out = n_classes - 1 if loss == "coral" else n_classes
    return timm.create_model(arch, pretrained=pretrained, num_classes=n_out)


def compute_loss(logits, y, loss, n_classes):
    if loss == "ce":
        return F.cross_entropy(logits, y)
    # CORAL: K-1 binary tasks "is y > k?"
    ks = torch.arange(n_classes - 1, device=y.device).unsqueeze(0)
    levels = (y.unsqueeze(1) > ks).float()
    return F.binary_cross_entropy_with_logits(logits, levels)


def to_probs(logits, loss):
    """Class probabilities for both head types (used for accuracy, ECE and Grad-CAM)."""
    if loss == "ce":
        return logits.softmax(1)
    c = torch.cummin(torch.sigmoid(logits), dim=1).values  # P(y > k), forced monotone
    probs = torch.cat([1 - c[:, :1], c[:, :-1] - c[:, 1:], c[:, -1:]], dim=1).clamp_min(0)
    return probs / probs.sum(1, keepdim=True).clamp_min(1e-8)


In [ ]:
%%writefile metrics.py
"""Metrics: accuracy, macro-F1, ordinal MAE, calibration (ECE), confusion matrix."""
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score


def ece_score(probs, y, bins=10):
    conf = probs.max(1)
    pred = probs.argmax(1)
    correct = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)


def summarize(probs, y, n_classes):
    pred = probs.argmax(1)
    return {
        "n": int(len(y)),
        "accuracy": float((pred == y).mean()),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(n_classes)), zero_division=0)),
        "mae": float(np.abs(pred - y).mean()),  # average class distance: ordinal "how far off"
        "ece": ece_score(probs, y),
        "confusion": confusion_matrix(y, pred, labels=list(range(n_classes))).tolist(),
    }


In [ ]:
%%writefile augment.py
"""Transforms: standard training aug, underwater colour-cast aug, and probe transforms."""
import random

import torch
from torchvision import transforms as T

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class UnderwaterCast:
    """Random blue/green tint, haze and brightness change (tensor in [0,1], CxHxW).

    Simulates different water colour, depth and turbidity so the model cannot rely on
    the absolute colour of one dataset.
    """

    def __init__(self, p=0.7):
        self.p = p

    def __call__(self, x):
        if random.random() > self.p:
            return x
        gain = torch.tensor([random.uniform(0.6, 1.0), random.uniform(0.8, 1.1),
                             random.uniform(0.9, 1.2)]).view(3, 1, 1)
        x = x * gain
        t = random.uniform(0.0, 0.35)
        haze = torch.tensor([random.uniform(0.1, 0.4), random.uniform(0.4, 0.7),
                             random.uniform(0.5, 0.8)]).view(3, 1, 1)
        x = (1 - t) * x + t * haze
        x = x * random.uniform(0.7, 1.2)
        return x.clamp(0, 1)


class Grayscale3:
    """Remove colour, keep structure."""

    def __call__(self, x):
        return x.mean(0, keepdim=True).repeat(3, 1, 1)


class PatchShuffle:
    """Shuffle a grid of patches: destroys shape/structure, keeps colour statistics."""

    def __init__(self, grid=4):
        self.grid = grid

    def __call__(self, x):
        _, H, W = x.shape
        g = self.grid
        h, w = H // g, W // g
        patches = [x[:, i * h:(i + 1) * h, j * w:(j + 1) * w] for i in range(g) for j in range(g)]
        random.shuffle(patches)
        out = x.clone()
        k = 0
        for i in range(g):
            for j in range(g):
                out[:, i * h:(i + 1) * h, j * w:(j + 1) * w] = patches[k]
                k += 1
        return out


def build_transforms(train, size=224, cast=False, probe=None):
    """probe: None | 'grayscale' | 'patch_shuffle' (evaluation-time stress tests)."""
    if train:
        ops = [T.RandomResizedCrop(size, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor()]
        if cast:
            ops.append(UnderwaterCast())
    else:
        ops = [T.Resize((size, size)), T.ToTensor()]
        if probe == "grayscale":
            ops.append(Grayscale3())
        elif probe == "patch_shuffle":
            ops.append(PatchShuffle())
    ops.append(T.Normalize(MEAN, STD))
    return T.Compose(ops)


In [ ]:
%%writefile data.py
"""Dataset utilities.

Every dataset is a folder tree where the *parent folder of each image* is its class,
e.g. data/roboflow/train/bleached_coral/img1.jpg. Roboflow/Kaggle exports with
train/valid/test sub-folders work as they are: all splits are merged and re-split here
with our own seed so that results are reproducible.
"""
import hashlib
import random
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Folder-name aliases -> canonical class. Add yours here if a dataset uses other names.
ALIASES = {
    "healthy": "healthy", "healthy_coral": "healthy", "healthy coral": "healthy",
    "unbleached": "healthy", "normal": "healthy",
    "bleached": "bleached", "bleached_coral": "bleached", "bleached coral": "bleached",
    "bleaching": "bleached",
    "dead": "dead", "dead_coral": "dead", "dead coral": "dead",
    # NOAA PIFSC dataset folder names
    "coral": "healthy", "coral_bl": "bleached",
    # other Roboflow and Kaggle naming variants
    "healthy_corals": "healthy", "healthy corals": "healthy", "not_bleached": "healthy",
    "bleached_corals": "bleached", "bleached corals": "bleached", "dead corals": "dead", "dead_corals": "dead",
}

# Ordinal order: healthy < bleached < dead
SCHEMES = {
    "3class": {"healthy": 0, "bleached": 1, "dead": 2},
    "binary": {"healthy": 0, "bleached": 1, "dead": 1},
}


def class_names(scheme):
    return ["healthy", "bleached", "dead"] if scheme == "3class" else ["healthy", "unhealthy"]


def scan_dataset(root, scheme="3class"):
    """Return [(path, label)] for every image whose parent folder is a known class."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset folder not found: {root}")
    items, skipped = [], Counter()
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in IMG_EXT:
            continue
        canon = ALIASES.get(p.parent.name.lower().strip())
        if canon is None:
            skipped[p.parent.name] += 1
            continue
        items.append((str(p), SCHEMES[scheme][canon]))
    if skipped:
        print(f"[data] {root.name}: skipped folders {dict(skipped)} (not in ALIASES)")
    if not items:
        raise RuntimeError(f"No usable images found in {root}. Check folder names vs ALIASES in data.py")
    return items


def dedupe(items):
    """Drop byte-identical duplicates (exports with augmented copies leak across splits otherwise)."""
    seen, out = set(), []
    for path, label in items:
        h = hashlib.md5(Path(path).read_bytes()).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        out.append((path, label))
    if len(out) != len(items):
        print(f"[data] removed {len(items) - len(out)} exact duplicates")
    return out


def source_group(path):
    """Source-image id, so crops/augmented copies of one photo never straddle train and test.

    Recognised: '<id>__<anything>.jpg' (patches made by download_datasets.py) and
    Roboflow exports '<id>.rf.<hash>.jpg' (augmented copies). Returns None otherwise.
    """
    name = Path(path).name
    if ".rf." in name:
        return name.split(".rf.")[0]
    stem = Path(path).stem
    if "__" in stem:
        return stem.split("__")[0]
    return None


def _split_grouped(items, seed, fracs):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for it in items:
        groups[source_group(it[0]) or it[0]].append(it)
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(items)
    train, val, test = [], [], []
    for k in keys:
        if len(train) < fracs[0] * n:
            train += groups[k]
        elif len(val) < fracs[1] * n:
            val += groups[k]
        else:
            test += groups[k]
    return train, val, test


def split_items(items, seed=0, fracs=(0.7, 0.15, 0.15)):
    """Train/val/test split. Group-aware when files carry a source-image id, else stratified."""
    if any(source_group(p) for p, _ in items):
        return _split_grouped(items, seed, fracs)
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for it in items:
        by_class[it[1]].append(it)
    train, val, test = [], [], []
    for _, group in sorted(by_class.items()):
        rng.shuffle(group)
        n = len(group)
        a = int(n * fracs[0])
        b = a + int(n * fracs[1])
        train += group[:a]
        val += group[a:b]
        test += group[b:]
    return train, val, test


def class_counts(items, n_classes):
    c = Counter(l for _, l in items)
    return [c.get(i, 0) for i in range(n_classes)]


class CoralDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


In [ ]:
%%writefile download_datasets.py
"""Download real coral datasets into data/<name>/<class>/ (the layout train.py reads).

  python download_datasets.py --list
  python download_datasets.py noaa                 # healthy vs bleached patches (Hawaii)
  python download_datasets.py noaa coralscapes     # + whole-reef photos (Red Sea), labels from masks
  ROBOFLOW_API_KEY=... python download_datasets.py roboflow_reef roboflow_health

Datasets
  noaa             NOAA PIFSC ESD, Hugging Face De129472/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset
                   (10,419 patches 224x224, CORAL / CORAL_BL, Hawaii 2014-2019). No key needed.
  coralscapes      Hugging Face EPFL-ECEO/coralscapes (2,075 reef photos, expert segmentation masks,
                   Apache-2.0). Healthy/bleached labels are DERIVED here from the masks (see below).
  roboflow_reef    Roboflow Universe "coral-reef-image-classification" (bleached/dead/healthy).
  roboflow_health  Roboflow Universe "coral-health-image-classifier-bleached-or-healthy" (CC BY 4.0).
                   Roboflow needs a free account + API key and the project forked into your workspace.

Coralscapes label rule (mask ids from the dataset card):
  alive coral   : 6, 17, 22, 25, 28, 31, 34, 36
  bleached coral: 4, 16, 19, 33
  dead coral    : 3, 20, 23, 32, 37   (dropped unless --keep-dead, so it matches NOAA's two classes)
  A patch/image gets a label only if coral covers >= --min-coral of it and one coral type makes up
  >= --purity of the coral pixels; mixed ones are skipped. Check "mask values seen" in the output.
"""
import argparse
import os
import random
from pathlib import Path

import numpy as np
from PIL import Image

HF_NOAA = "De129472/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
HF_CORALSCAPES = "EPFL-ECEO/coralscapes"
CS_ALIVE = [6, 17, 22, 25, 28, 31, 34, 36]
CS_BLEACHED = [4, 16, 19, 33]
CS_DEAD = [3, 20, 23, 32, 37]

ROBOFLOW = {
    "roboflow_reef": ("project-idpmi", "coral-reef-image-classification"),
    "roboflow_health": ("techwiz-wbywj", "coral-health-image-classifier-bleached-or-healthy"),
}
ALL = ["noaa", "coralscapes"] + list(ROBOFLOW)


# ------------------------------------------------------------------ summary
def summarize_folder(name, root):
    """Use the project's own loader so we see exactly what train.py will see."""
    from data import scan_dataset
    try:
        items = scan_dataset(Path(root) / name, "3class")
    except Exception as e:  # noqa: BLE001
        print(f"[{name}] PROBLEM: {e}")
        return False
    counts = {}
    for _, l in items:
        counts[l] = counts.get(l, 0) + 1
    names = ["healthy", "bleached", "dead"]
    print(f"[{name}] ready: {len(items)} images, " + ", ".join(f"{names[k]}={v}" for k, v in sorted(counts.items())))
    return True


# ------------------------------------------------------------------ NOAA
def get_noaa(root, **_):
    out = Path(root) / "noaa"
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HF_NOAA, repo_type="dataset", local_dir=str(out))
    imgs = [p for p in out.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
    if not imgs:  # files came as parquet: decode them into class folders
        print("[noaa] no image files found, decoding parquet via `datasets`")
        from datasets import load_dataset
        ds = load_dataset(HF_NOAA)
        for split in ds:
            names = ds[split].features["label"].names
            for i, ex in enumerate(ds[split]):
                d = out / split / names[ex["label"]]
                d.mkdir(parents=True, exist_ok=True)
                ex["image"].convert("RGB").save(d / f"{i}.png")
    return summarize_folder("noaa", root)


# ------------------------------------------------------------------ Coralscapes
def _label_from_counts(a, b, d, total, min_coral, purity, keep_dead):
    coral = a + b + d
    if coral < min_coral * total or coral == 0:
        return None
    if a / coral >= purity:
        return "healthy"
    if b / coral >= purity:
        return "bleached"
    if keep_dead and d / coral >= purity:
        return "dead"
    return None


def _mask_array(label_img, offset):
    m = np.asarray(label_img)
    if m.ndim == 3:
        raise ValueError("Coralscapes mask is RGB; expected a class-index mask. "
                         "Open one mask and tell the maintainer / adjust _mask_array().")
    return m.astype(np.int32) + offset


def convert_coralscapes(rows, out_dir, mode="patch", tile=256, min_coral=0.5, purity=0.8, keep_dead=False,
                        max_per_class_per_image=3, max_per_class=2500, offset=0, seed=0):
    """rows: iterable of (tag, PIL image, PIL mask). Writes <out_dir>/<class>/<tag>__rXcY.jpg."""
    rng = random.Random(seed)
    out = Path(out_dir)
    classes = ["healthy", "bleached"] + (["dead"] if keep_dead else [])
    for c in classes:
        (out / c).mkdir(parents=True, exist_ok=True)
    saved = {c: 0 for c in classes}
    seen_values = set()
    for tag, img, mask in rows:
        m = _mask_array(mask, offset)
        if len(seen_values) < 60:
            seen_values |= set(np.unique(m).tolist())
        img = img.convert("RGB")
        cand = {c: [] for c in classes}
        if mode == "image":
            a = np.isin(m, CS_ALIVE).sum()
            b = np.isin(m, CS_BLEACHED).sum()
            d = np.isin(m, CS_DEAD).sum()
            lab = _label_from_counts(a, b, d, m.size, min_coral=0.1, purity=purity, keep_dead=keep_dead)
            if lab:
                cand[lab].append((None, None))
        else:
            H, W = m.shape
            for r in range(0, H - tile + 1, tile):
                for c in range(0, W - tile + 1, tile):
                    t = m[r:r + tile, c:c + tile]
                    a = np.isin(t, CS_ALIVE).sum()
                    b = np.isin(t, CS_BLEACHED).sum()
                    d = np.isin(t, CS_DEAD).sum()
                    lab = _label_from_counts(a, b, d, t.size, min_coral, purity, keep_dead)
                    if lab:
                        cand[lab].append((r, c))
        for lab, spots in cand.items():
            rng.shuffle(spots)
            for r, c in spots[:max_per_class_per_image]:
                if saved[lab] >= max_per_class:
                    break
                if r is None:
                    im = img.copy()
                    im.thumbnail((768, 768))
                    name = f"{tag}.jpg"
                else:
                    im = img.crop((c, r, c + tile, r + tile))
                    name = f"{tag}__r{r}c{c}.jpg"
                im.save(out / lab / name, quality=92)
                saved[lab] += 1
        if all(v >= max_per_class for v in saved.values()):
            break
    print(f"[coralscapes] mask values seen (first images): {sorted(seen_values)[:45]}")
    print(f"[coralscapes] saved {saved}")
    if saved["healthy"] == 0 or saved["bleached"] == 0:
        print("[coralscapes] WARNING: a class is empty. Mask ids may be shifted: try --coralscapes-offset 1 "
              "(or -1), and compare 'mask values seen' with the id table in this file's header.")
    return saved


def _coralscapes_rows(max_images):
    from datasets import load_dataset
    n = 0
    for split in ("train", "validation", "test"):
        for i, ex in enumerate(load_dataset(HF_CORALSCAPES, split=split, streaming=True)):
            if max_images and n >= max_images:
                return
            yield f"{split}{i:04d}", ex["image"], ex["label"]
            n += 1


def get_coralscapes(root, max_images=800, mode="patch", keep_dead=False, offset=0, min_coral=0.5,
                    purity=0.8, seed=0, **_):
    out = Path(root) / f"coralscapes_{mode}"
    convert_coralscapes(_coralscapes_rows(max_images), out, mode=mode, keep_dead=keep_dead, offset=offset,
                        min_coral=min_coral, purity=purity, seed=seed)
    return summarize_folder(out.name, root)


# ------------------------------------------------------------------ Roboflow
def get_roboflow(name, root, version=None, rf_workspace=None, **_):
    key = os.environ.get("ROBOFLOW_API_KEY")
    if not key:
        print(f"[{name}] skipped: set ROBOFLOW_API_KEY (free account) and fork the project first. "
              "Or download the 'Folder Structure' zip by hand into data/" + name + "/")
        return False
    from roboflow import Roboflow
    rf = Roboflow(api_key=key)
    orig_ws, project = ROBOFLOW[name]
    out = Path(root) / name
    for ws in (rf_workspace, None, orig_ws):  # your fork first, then the original
        try:
            ws_obj = rf.workspace(ws) if ws else rf.workspace()
            proj = ws_obj.project(project)
            v = version
            if v is None:
                v = max(int(str(getattr(x, "version", x)).split("/")[-1]) for x in proj.versions())
            proj.version(int(v)).download("folder", location=str(out))
            return summarize_folder(name, root)
        except Exception as e:  # noqa: BLE001
            print(f"[{name}] workspace {ws or '(default)'} failed: {type(e).__name__}: {str(e)[:150]}")
    print(f"[{name}] could not download. Fork the project on roboflow.com, then download 'Folder Structure' manually.")
    return False


# ------------------------------------------------------------------ CLI
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("names", nargs="*", help=f"any of: {', '.join(ALL)}, or 'all'")
    ap.add_argument("--list", action="store_true")
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--coralscapes-mode", choices=["patch", "image"], default="patch")
    ap.add_argument("--max-images", type=int, default=800, help="Coralscapes photos to read (0 = all)")
    ap.add_argument("--keep-dead", action="store_true")
    ap.add_argument("--coralscapes-offset", type=int, default=0, help="add to mask values if ids are shifted")
    ap.add_argument("--min-coral", type=float, default=0.5)
    ap.add_argument("--purity", type=float, default=0.8)
    ap.add_argument("--roboflow-version", type=int, default=None)
    ap.add_argument("--roboflow-workspace", default=None, help="your workspace slug if the fork is not the default")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    if a.list or not a.names:
        print(__doc__)
        return
    names = ALL if "all" in a.names else a.names
    for n in names:
        if n not in ALL:
            raise SystemExit(f"unknown dataset '{n}'. Choose from: {', '.join(ALL)}")
    for n in names:
        print(f"\n=== {n} ===")
        if n == "noaa":
            get_noaa(a.data_root)
        elif n == "coralscapes":
            get_coralscapes(a.data_root, a.max_images, a.coralscapes_mode, a.keep_dead, a.coralscapes_offset,
                            a.min_coral, a.purity, a.seed)
        else:
            get_roboflow(n, a.data_root, a.roboflow_version, a.roboflow_workspace)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""Train on source dataset(s), then evaluate in-domain and on unseen target dataset(s).

Example:
  python train.py --data-root data --source roboflow --targets kaggle,kohtao \
      --arch efficientnet_b0 --loss ce --epochs 15 --out runs/effb0_ce
"""
import argparse
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from augment import build_transforms
from data import CoralDataset, class_counts, class_names, dedupe, scan_dataset, split_items
from metrics import summarize
from models import build_model, compute_loss, to_probs


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


@torch.no_grad()
def predict(model, loader, device, loss):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        P.append(to_probs(model(x.to(device)).float(), loss).cpu())
        Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def make_loader(items, size, bs, workers, train=False, cast=False, sampler=None, probe=None):
    ds = CoralDataset(items, build_transforms(train, size, cast, probe))
    return DataLoader(ds, batch_size=bs, shuffle=train and sampler is None, sampler=sampler,
                      num_workers=workers, pin_memory=torch.cuda.is_available())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--source", required=True, help="comma-separated dataset folder names to train on")
    ap.add_argument("--targets", default="", help="comma-separated unseen datasets for cross-dataset test")
    ap.add_argument("--arch", default="efficientnet_b0")
    ap.add_argument("--loss", choices=["ce", "coral"], default="ce")
    ap.add_argument("--scheme", choices=["3class", "binary"], default="3class")
    ap.add_argument("--cast", action="store_true", help="underwater colour-cast augmentation")
    ap.add_argument("--balanced-sampler", action="store_true")
    ap.add_argument("--no-pretrained", dest="pretrained", action="store_false")
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--size", type=int, default=224)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", default="runs/run")
    args = ap.parse_args()

    set_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    names = class_names(args.scheme)
    K = len(names)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    sources = [s for s in args.source.split(",") if s]
    targets = [t for t in args.targets.split(",") if t]

    # --- data: split each dataset on its own, so every source keeps its own test portion
    splits, train_items, val_items = {}, [], []
    for n in dict.fromkeys(sources + targets):
        items = dedupe(scan_dataset(Path(args.data_root) / n, args.scheme))
        splits[n] = split_items(items, args.seed)
        print(f"[data] {n}: {len(items)} images, class counts {class_counts(items, K)}")
    for n in sources:
        train_items += splits[n][0]
        val_items += splits[n][1]

    eval_sets = {}
    for n in sources:
        eval_sets[f"{n} (in-domain test)"] = splits[n][2]
    for n in targets:
        # a target that is also a source is only evaluated on its held-out test split
        eval_sets[f"{n} (cross-dataset)"] = splits[n][2] if n in sources else sum(splits[n], [])

    sampler = None
    if args.balanced_sampler:
        cnt = class_counts(train_items, K)
        w = [1.0 / max(cnt[l], 1) for _, l in train_items]
        sampler = WeightedRandomSampler(w, len(w), replacement=True)

    train_loader = make_loader(train_items, args.size, args.bs, args.workers, True, args.cast, sampler)
    val_loader = make_loader(val_items, args.size, args.bs, args.workers)

    # --- model
    model = build_model(args.arch, K, args.loss, args.pretrained).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

    best_f1, best_state = -1.0, None
    for ep in range(1, args.epochs + 1):
        model.train()
        run = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type=device, enabled=device == "cuda"):
                logits = model(x)
            loss = compute_loss(logits.float(), y, args.loss, K)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run += loss.item() * len(y)
        sched.step()
        P, Y = predict(model, val_loader, device, args.loss)
        m = summarize(P, Y, K)
        print(f"epoch {ep:02d} loss {run / len(train_items):.4f} val_acc {m['accuracy']:.3f} val_f1 {m['macro_f1']:.3f}")
        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save({"state": best_state, "arch": args.arch, "loss": args.loss, "scheme": args.scheme,
                "size": args.size, "classes": names}, out / "model.pt")

    # --- evaluation: in-domain vs cross-dataset (the headline comparison)
    results = {}
    for name, items in eval_sets.items():
        P, Y = predict(model, make_loader(items, args.size, args.bs, args.workers), device, args.loss)
        results[name] = summarize(P, Y, K)
        r = results[name]
        print(f"[eval] {name}: n={r['n']} acc={r['accuracy']:.3f} f1={r['macro_f1']:.3f} "
              f"mae={r['mae']:.3f} ece={r['ece']:.3f}")

    (out / "results.json").write_text(json.dumps({"args": vars(args), "best_val_f1": best_f1,
                                                  "eval": results}, indent=2))
    (out / "items.json").write_text(json.dumps({"train": train_items, "eval": eval_sets}))
    print(f"saved to {out}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze.py
"""Shortcut analysis for a trained run.

  python analyze.py --run runs/effb0_ce --task probes       # grayscale / patch-shuffle stress tests
  python analyze.py --run runs/effb0_ce --task colour       # colour-histogram-only baseline
  python analyze.py --run runs/effb0_ce --task gradcam      # attention overlays

Reading the results:
  * accuracy collapses in grayscale        -> model relies on colour
  * accuracy survives patch-shuffle        -> model uses colour statistics, not coral structure
  * colour-only baseline ~ CNN accuracy    -> the "detection" is mostly colour
"""
import argparse
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression

from augment import MEAN, STD, build_transforms
from metrics import summarize
from models import build_model, to_probs
from train import make_loader, predict


def load_run(run, device):
    ckpt = torch.load(Path(run) / "model.pt", map_location=device)
    K = len(ckpt["classes"])
    model = build_model(ckpt["arch"], K, ckpt["loss"], pretrained=False)
    model.load_state_dict(ckpt["state"])
    model.to(device).eval()
    items = json.loads((Path(run) / "items.json").read_text())
    return model, ckpt, items


def task_probes(model, ckpt, items, device, run, workers):
    K = len(ckpt["classes"])
    table = {}
    for name, ev in items["eval"].items():
        table[name] = {}
        for probe in [None, "grayscale", "patch_shuffle"]:
            loader = make_loader(ev, ckpt["size"], 32, workers, probe=probe)
            P, Y = predict(model, loader, device, ckpt["loss"])
            table[name][probe or "normal"] = summarize(P, Y, K)
    print(f"{'dataset':38s} {'normal':>8s} {'grayscale':>10s} {'shuffle':>8s}   (macro-F1)")
    for name, r in table.items():
        print(f"{name:38s} {r['normal']['macro_f1']:8.3f} {r['grayscale']['macro_f1']:10.3f} "
              f"{r['patch_shuffle']['macro_f1']:8.3f}")
    (Path(run) / "probes.json").write_text(json.dumps(table, indent=2))


def colour_features(src, bins=16):
    """src: image path or PIL image."""
    img = src if isinstance(src, Image.Image) else Image.open(src)
    a = np.asarray(img.convert("RGB").resize((64, 64)), dtype=np.float32) / 255.0
    hist = [np.histogram(a[..., c], bins=bins, range=(0, 1))[0] / (64 * 64) for c in range(3)]
    return np.concatenate(hist + [a.mean((0, 1)), a.std((0, 1))])


def task_colour(ckpt, items, run):
    K = len(ckpt["classes"])
    Xtr = np.stack([colour_features(p) for p, _ in items["train"]])
    ytr = np.array([l for _, l in items["train"]])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
    table = {}
    for name, ev in items["eval"].items():
        X = np.stack([colour_features(p) for p, _ in ev])
        y = np.array([l for _, l in ev])
        probs = np.zeros((len(y), K))
        probs[:, clf.classes_] = clf.predict_proba(X)
        table[name] = summarize(probs, y, K)
        print(f"[colour-only] {name}: acc={table[name]['accuracy']:.3f} f1={table[name]['macro_f1']:.3f}")
    (Path(run) / "colour_baseline.json").write_text(json.dumps(table, indent=2))


def target_layer(model, arch):
    if arch.startswith("resnet"):
        return model.layer4[-1]
    if arch.startswith("efficientnet"):
        return model.blocks[-1]
    if arch.startswith("convnext"):
        return model.stages[-1]
    raise ValueError(f"Grad-CAM layer not defined for {arch}; add it in target_layer()")


def gradcam(model, layer, x, loss):
    store = {}

    def fwd_hook(_, __, out):
        store["a"] = out
        out.register_hook(lambda g: store.__setitem__("g", g))

    h = layer.register_forward_hook(fwd_hook)
    model.zero_grad()
    probs = to_probs(model(x), loss)
    cls = probs.argmax(1)
    probs.gather(1, cls[:, None]).sum().backward()
    h.remove()
    w = store["g"].mean((2, 3), keepdim=True)
    cam = F.relu((w * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    mn, mx = cam.amin((1, 2), keepdim=True), cam.amax((1, 2), keepdim=True)
    cam = (cam - mn) / (mx - mn + 1e-8)
    return cam.detach().cpu().numpy(), cls.cpu().numpy(), probs.detach().cpu().numpy()


def task_gradcam(model, ckpt, items, device, run, per_set):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    layer = target_layer(model, ckpt["arch"])
    tf = build_transforms(False, ckpt["size"])
    names = ckpt["classes"]
    mean, std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
    sets = list(items["eval"].items())
    fig, axes = plt.subplots(len(sets), per_set, figsize=(3 * per_set, 3.2 * len(sets)), squeeze=False)
    rng = random.Random(0)
    for r, (name, ev) in enumerate(sets):
        for c, (path, label) in enumerate(rng.sample(ev, min(per_set, len(ev)))):
            x = tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
            cam, cls, probs = gradcam(model, layer, x, ckpt["loss"])
            img = (x[0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
            ax = axes[r][c]
            ax.imshow(img)
            ax.imshow(cam[0], cmap="jet", alpha=0.45)
            ax.set_title(f"pred {names[cls[0]]} ({probs[0, cls[0]]:.2f})\ntrue {names[label]}", fontsize=8)
            ax.axis("off")
        axes[r][0].set_ylabel(name, fontsize=8)
        axes[r][0].text(-0.05, 0.5, name, transform=axes[r][0].transAxes, rotation=90,
                        va="center", ha="right", fontsize=8)
    plt.tight_layout()
    out = Path(run) / "gradcam.png"
    plt.savefig(out, dpi=130)
    print(f"saved {out}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--task", choices=["probes", "colour", "gradcam"], required=True)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--per-set", type=int, default=6)
    args = ap.parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, ckpt, items = load_run(args.run, device)
    if args.task == "probes":
        task_probes(model, ckpt, items, device, args.run, args.workers)
    elif args.task == "colour":
        task_colour(ckpt, items, args.run)
    else:
        task_gradcam(model, ckpt, items, device, args.run, args.per_set)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile viz.py
"""Charts for the experiment results (shared by report.py and the live app).

Colour use: blue / orange / aqua are the first three slots of a validated categorical
palette (safe for colour-blind readers). Heat maps use a single-hue blue ramp.
Every chart has a legend or direct labels, and the full numbers are in the table view.
"""
import json
import textwrap
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402
from matplotlib.patches import Patch  # noqa: E402

SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SEQ = LinearSegmentedColormap.from_list("blue_seq", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "text.color": INK,
})


# ----------------------------------------------------------------------------- loading
def load_runs(runs_dir):
    runs = {}
    for d in sorted(Path(runs_dir).glob("*")):
        f = d / "results.json"
        if not f.exists():
            continue
        r = {"dir": d, "results": json.loads(f.read_text())}
        for key, fn in [("probes", "probes.json"), ("colour", "colour_baseline.json")]:
            p = d / fn
            r[key] = json.loads(p.read_text()) if p.exists() else None
        runs[d.name] = r
    return runs


def class_names(run):
    return ["healthy", "bleached", "dead"] if run["results"]["args"]["scheme"] == "3class" else ["healthy", "unhealthy"]


def _kind(eval_name):
    return "cross-dataset" if "cross-dataset" in eval_name else "in-domain"


def _style(ax, grid_axis="y"):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(colors=MUTED, length=0, labelsize=9)
    ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def _wrap(s, w=22):
    return textwrap.fill(s, w)


# ----------------------------------------------------------------------------- charts
def fig_overview(runs):
    """In-domain vs cross-dataset: the headline chart. One row per run."""
    names = list(runs)
    metrics = [("macro_f1", "Macro-F1 (higher is better)"),
               ("mae", "Class error, MAE (lower is better)"),
               ("ece", "Calibration error, ECE (lower is better)")]
    fig, axes = plt.subplots(1, 3, figsize=(13, max(3.0, 0.8 * len(names) + 1.9)), sharey=True)
    y = np.arange(len(names))
    h = 0.36
    for ax, (key, title) in zip(axes, metrics):
        allv = []
        for off, kind, color in [(-h / 2, "in-domain", BLUE), (h / 2, "cross-dataset", ORANGE)]:
            vals = np.array([
                np.mean([m[key] for k, m in runs[n]["results"]["eval"].items() if _kind(k) == kind] or [np.nan])
                for n in names])
            allv += list(vals[~np.isnan(vals)])
            ax.barh(y + off, np.nan_to_num(vals), height=h - 0.05, color=color, edgecolor=SURFACE, linewidth=2)
            for yy, v in zip(y + off, vals):
                if not np.isnan(v):
                    ax.text(v + 0.012, yy, f"{v:.2f}", va="center", fontsize=9, color=INK2)
        top = 1.0 if key == "macro_f1" else max(0.05, max(allv or [0.05]))
        ax.set_xlim(0, top * 1.18)
        ax.set_title(title, fontsize=10, color=INK, loc="left", pad=8)
        _style(ax, "x")
    axes[0].set_yticks(y)
    axes[0].set_yticklabels([_wrap(n, 20) for n in names], fontsize=9, color=INK2)
    axes[0].invert_yaxis()
    fig.legend(handles=[Patch(color=BLUE, label="In-domain test (same dataset as training)"),
                        Patch(color=ORANGE, label="Cross-dataset (unseen dataset)")],
               loc="upper right", frameon=False, ncol=2, fontsize=9, bbox_to_anchor=(0.99, 1.0))
    H = fig.get_figheight()
    fig.text(0.01, 1 - 0.28 / H, "Does accuracy survive a different dataset?", ha="left", va="center",
             fontsize=13, fontweight="bold")
    fig.text(0.01, 1 - 0.58 / H, "Each bar is the mean over that run's evaluation sets of that kind; per-dataset values are in the table.",
             ha="left", va="center", fontsize=8.5, color=MUTED)
    fig.tight_layout(rect=[0, 0, 1, 1 - 0.8 / H])
    return fig


def fig_confusion(name, run):
    sets = run["results"]["eval"]
    classes = class_names(run)
    n = len(sets)
    fig, axes = plt.subplots(1, n, figsize=(4.3 * n, 4.4), squeeze=False)
    for ax, (k, m) in zip(axes[0], sets.items()):
        cm = np.array(m["confusion"], float)
        norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(norm, cmap=SEQ, vmin=0, vmax=1)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, f"{int(cm[i, j])}\n{norm[i, j]:.0%}", ha="center", va="center", fontsize=10,
                        color="#ffffff" if norm[i, j] > 0.5 else INK)
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes, fontsize=9, color=INK2)
        ax.set_yticklabels(classes, fontsize=9, color=INK2)
        ax.set_xlabel("Predicted", fontsize=9, color=MUTED)
        ax.set_ylabel("True", fontsize=9, color=MUTED)
        ax.set_title(f"{_wrap(k, 28)}\nacc {m['accuracy']:.2f} · F1 {m['macro_f1']:.2f}", fontsize=9.5, color=INK, loc="left")
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
    fig.suptitle(f"Confusion matrices: {name}   (cell = count and share of the true class)", x=0.01, ha="left",
                 fontsize=11, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    return fig


def _grouped(ax, groups, series, colors, labels):
    x = np.arange(len(groups))
    w = 0.8 / len(series)
    for i, (vals, c) in enumerate(zip(series, colors)):
        pos = x + (i - (len(series) - 1) / 2) * w
        ax.bar(pos, vals, width=w - 0.04, color=c, edgecolor=SURFACE, linewidth=2)
        for p, v in zip(pos, vals):
            ax.text(p, v + 0.015, f"{v:.2f}", ha="center", fontsize=8.5, color=INK2)
    ax.set_xticks(x)
    ax.set_xticklabels([_wrap(g, 22) for g in groups], fontsize=9, color=INK2)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Macro-F1", fontsize=9, color=MUTED)
    _style(ax, "y")
    ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(colors, labels)], frameon=False, fontsize=9,
              loc="lower left", ncol=len(series), bbox_to_anchor=(0.0, 1.0), borderaxespad=0.3)


def fig_probes(name, run):
    pr = run["probes"]
    groups = list(pr)
    series = [[pr[g][k]["macro_f1"] for g in groups] for k in ("normal", "grayscale", "patch_shuffle")]
    fig, ax = plt.subplots(figsize=(max(6.5, 2.8 * len(groups)), 4.4))
    _grouped(ax, groups, series, [BLUE, ORANGE, AQUA],
             ["Normal", "Grayscale (colour removed)", "Patch-shuffled (shape destroyed)"])
    fig.suptitle(f"Stress tests: {name}", x=0.01, ha="left", fontsize=11, fontweight="bold")
    fig.text(0.01, 0.01, "Big drop in grayscale = relies on colour. Small drop when shuffled = relies on colour statistics, not coral shape.",
             fontsize=8, color=MUTED)
    fig.tight_layout(rect=[0, 0.04, 1, 0.93])
    return fig


def fig_colour(name, run):
    co = run["colour"]
    groups = list(co)
    cnn = [run["results"]["eval"][g]["macro_f1"] for g in groups]
    col = [co[g]["macro_f1"] for g in groups]
    fig, ax = plt.subplots(figsize=(max(6.0, 2.6 * len(groups)), 4.4))
    _grouped(ax, groups, [cnn, col], [BLUE, ORANGE], ["CNN", "Colour-only baseline (logistic regression)"])
    fig.suptitle(f"CNN vs colour-only baseline: {name}", x=0.01, ha="left", fontsize=11, fontweight="bold")
    fig.text(0.01, 0.01, "If the colour-only baseline is close to the CNN, the task is mostly solvable from colour alone.",
             fontsize=8, color=MUTED)
    fig.tight_layout(rect=[0, 0.04, 1, 0.93])
    return fig


def results_table(runs):
    rows = []
    for n, r in runs.items():
        for k, m in r["results"]["eval"].items():
            rows.append({
                "Run": n, "Evaluation set": k, "Type": _kind(k), "Images": m["n"],
                "Accuracy": m["accuracy"], "Macro-F1": m["macro_f1"], "MAE": m["mae"], "ECE": m["ece"],
                "Colour-only F1": r["colour"][k]["macro_f1"] if r["colour"] and k in r["colour"] else np.nan,
                "Grayscale F1": r["probes"][k]["grayscale"]["macro_f1"] if r["probes"] and k in r["probes"] else np.nan,
                "Shuffled F1": r["probes"][k]["patch_shuffle"]["macro_f1"] if r["probes"] and k in r["probes"] else np.nan,
            })
    return pd.DataFrame(rows).round(3)


# ----------------------------------------------------------------------------- report
def save_all(runs_dir, out_dir):
    runs = load_runs(runs_dir)
    if not runs:
        raise SystemExit(f"No finished runs (results.json) found in {runs_dir}/. Train first.")
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    sections = []

    def save(fig, fname, title, alt):
        fig.savefig(out / fname, dpi=130, bbox_inches="tight")
        plt.close(fig)
        sections.append((title, fname, alt))

    save(fig_overview(runs), "overview.png", "Overview: in-domain vs cross-dataset",
         "Bar charts of macro-F1, MAE and ECE for every run, in-domain versus cross-dataset")
    for n, r in runs.items():
        save(fig_confusion(n, r), f"confusion_{n}.png", f"Confusion matrices: {n}", f"Confusion matrices for run {n}")
        if r["probes"]:
            save(fig_probes(n, r), f"probes_{n}.png", f"Stress tests: {n}", f"Macro-F1 under grayscale and patch-shuffle for run {n}")
        if r["colour"]:
            save(fig_colour(n, r), f"colour_{n}.png", f"CNN vs colour-only: {n}", f"CNN versus colour-only baseline for run {n}")
        g = r["dir"] / "gradcam.png"
        if g.exists():
            (out / f"gradcam_{n}.png").write_bytes(g.read_bytes())
            sections.append((f"Grad-CAM attention: {n}", f"gradcam_{n}.png", f"Grad-CAM heat maps for run {n}"))

    df = results_table(runs)
    df.to_csv(out / "results_table.csv", index=False)
    body = "".join(f'<section><h2>{t}</h2><img src="{f}" alt="{a}"></section>' for t, f, a in sections)
    html = f"""<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Coral bleaching robustness report</title>
<style>body{{font:15px system-ui,-apple-system,"Segoe UI",sans-serif;background:#f9f9f7;color:#0b0b0b;margin:0;padding:24px;max-width:1100px;margin:auto}}
section{{background:#fcfcfb;border:1px solid #e1e0d9;border-radius:10px;padding:16px;margin:16px 0}}img{{max-width:100%}}
table{{border-collapse:collapse;font-size:13px;width:100%}}th,td{{border-bottom:1px solid #e1e0d9;padding:6px 8px;text-align:left}}th{{color:#52514e}}</style>
<h1>Coral bleaching robustness: results</h1>
<section><h2>Results table (all numbers)</h2>{df.to_html(index=False, na_rep="–", border=0)}</section>{body}</html>"""
    (out / "index.html").write_text(html, encoding="utf-8")
    print(f"report written to {out}/index.html ({len(sections)} charts)")


In [ ]:
%%writefile report.py
"""Build all charts and an HTML report from the finished runs.

  python report.py --runs runs --out report      # then open report/index.html
"""
import argparse

import viz

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", default="runs")
    ap.add_argument("--out", default="report")
    a = ap.parse_args()
    viz.save_all(a.runs, a.out)


In [ ]:
# 3. Automatically download real NOAA Hawaii Coral Bleaching Dataset (10,419 images)
# No manual searching or Kaggle inputs needed!
!python download_datasets.py noaa


In [ ]:
# 4. Train Deep Learning Models on GPU (EfficientNet-B0)
print('=== Training Baseline Model ===')
!python train.py --data-root data --source noaa --scheme binary --arch efficientnet_b0 --epochs 12 --bs 64 --out runs/effb0_noaa

print('\n=== Training Model with Underwater Colour-Cast Augmentation (Fix) ===')
!python train.py --data-root data --source noaa --scheme binary --cast --arch efficientnet_b0 --epochs 12 --bs 64 --out runs/effb0_noaa_cast


In [ ]:
# 5. Run Shortcut Probes & Robustness Analysis
# (Colour baseline, Grayscale probe, Patch-shuffle probe, Grad-CAM heatmaps)
for run_name in ['runs/effb0_noaa', 'runs/effb0_noaa_cast']:
    print(f'\n=== Analyzing {run_name} ===')
    !python analyze.py --run "{run_name}" --task colour
    !python analyze.py --run "{run_name}" --task probes --workers 2
    !python analyze.py --run "{run_name}" --task gradcam


In [ ]:
# 6. Generate HTML report & charts
!python report.py --runs runs --out report

from IPython.display import Image, display
print('=== Benchmark Summary Overview ===')
from pathlib import Path
if Path('report/overview.png').exists():
    display(Image(filename='report/overview.png'))


In [ ]:
# 7. Zip all trained checkpoints, results, and charts for download
!zip -r coral_bleaching_results.zip runs report

from IPython.display import FileLink, display
print('=' * 70)
print('Training complete! Download your zipped models and report below:')
print("Or download coral_bleaching_results.zip from the Output tab on Kaggle's right sidebar.")
print('=' * 70)
display(FileLink('coral_bleaching_results.zip'))
